# Nav2 NavigateThroughPoses Helper

Load exported SE(3) poses, send them to Nav2's `NavigateThroughPoses` action, or just place them on the BT navigator's blackboard. Works with ROS 2 Humble (make sure Nav2 is running).

In [1]:
import os
import json
from pathlib import Path
from typing import List

import rclpy
from rclpy.action import ActionClient
from rclpy.node import Node
from geometry_msgs.msg import PoseStamped
from nav2_msgs.action import NavigateThroughPoses
from builtin_interfaces.msg import Duration


In [2]:
# Configuration
poses_file = Path('../TrajectoryTools/ExampleTrajectories/Roboalm/exports/box0.json')
nav2_action_name = '/navigate_through_poses'
default_frame = 'map'
wait_for_server_timeout = 10.0  # seconds
# Optional: set ROS_DOMAIN_ID or other env vars here
# os.environ['ROS_DOMAIN_ID'] = '42'


In [3]:
def load_poses(file_path: Path, fallback_frame: str) -> List[PoseStamped]:
    data = json.loads(file_path.read_text())
    frame_id = data.get('frame_id', fallback_frame)
    poses = []
    for entry in data.get('poses', []):
        pose = PoseStamped()
        pose.header.frame_id = entry.get('frame_id', frame_id)
        pose.pose.position.x = entry['position']['x']
        pose.pose.position.y = entry['position']['y']
        pose.pose.position.z = entry['position']['z']
        pose.pose.orientation.x = entry['orientation']['x']
        pose.pose.orientation.y = entry['orientation']['y']
        pose.pose.orientation.z = entry['orientation']['z']
        pose.pose.orientation.w = entry['orientation']['w']
        poses.append(pose)
    return poses


In [4]:
class NavThroughPosesClient(Node):
    def __init__(self, action_name: str):
        super().__init__('notebook_nav_through_poses_client')
        self._client = ActionClient(self, NavigateThroughPoses, action_name)

    def send_goal(self, poses: List[PoseStamped]):
        if not poses:
            raise ValueError('No poses to send')
        if not self._client.wait_for_server(timeout_sec=wait_for_server_timeout):
            raise RuntimeError('NavigateThroughPoses action server not available')
        goal = NavigateThroughPoses.Goal()
        goal.poses = poses
        goal.behavior_tree = ''
        goal.speed = 0.0
        goal.time_allowance = Duration(sec=0)
        self.get_logger().info(f'Sending {len(poses)} poses to {self._client._action_name}')
        return self._client.send_goal_async(goal)


In [5]:
def send_nav_goal(file_path: Path):
    poses = load_poses(file_path, default_frame)
    if not poses:
        raise ValueError('No poses found in file')
    rclpy.init()
    node = NavThroughPosesClient(nav2_action_name)
    future = node.send_goal(poses)
    rclpy.spin_until_future_complete(node, future)
    goal_handle = future.result()
    if not goal_handle.accepted:
        node.get_logger().error('Goal rejected')
        node.destroy_node()
        rclpy.shutdown()
        return
    node.get_logger().info('Goal accepted, waiting for result...')
    result_future = goal_handle.get_result_async()
    rclpy.spin_until_future_complete(node, result_future)
    result = result_future.result()
    node.get_logger().info(f'Result status: {result.status}')
    node.destroy_node()
    rclpy.shutdown()


## Send poses to Nav2 (populates BT blackboard)

Run the following cell after Nav2 is up. The behavior tree's blackboard will receive the goal sequence automatically once the NavigateThroughPoses action is dispatched.

In [6]:
send_nav_goal(poses_file)

KeyboardInterrupt: 